<a href="https://colab.research.google.com/github/jihandong/MLCExercises/blob/main/mlc_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Compiler Exercises

## Install the Environment




In [ ]:
%pip install -q "apache-tvm==0.26.0" "apache-tvm-ffi==0.1.16"

In [ ]:
import numpy as np
import tvm
from tvm.script import tirx as T

dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
c_mm_relu = np.maximum(a_np @ b_np, 0)

@tvm.script.ir_module
class MyModule:
    @T.prim_func(s_tir=True)
    def mm_relu(A: T.Buffer((128, 128), "float32"),
                B: T.Buffer((128, 128), "float32"),
                C: T.Buffer((128, 128), "float32")):
        T.func_attr({"global_symbol": "mm_relu", "tirx.noalias": True})
        Y = T.sblock_alloc_buffer((128, 128), dtype="float32")
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


## 2.4.5.1 J1 Profile

In [ ]:
from statistics import median


def transform(mod, j_factor):
    """Apply the loop transform from section 2.4.5.1 to a fresh schedule."""
    schedule = tvm.s_tir.Schedule(mod)
    matmul = schedule.get_sblock("Y", func_name="mm_relu")
    relu = schedule.get_sblock("C", func_name="mm_relu")
    _, j, k = schedule.get_loops(matmul)
    j_outer, j_inner = schedule.split(j, factors=[None, j_factor])
    schedule.reorder(j_outer, k, j_inner)
    schedule.reverse_compute_at(relu, j_outer)
    return schedule.mod


dev = tvm.cpu(0)
a_nd = tvm.runtime.tensor(a_np, device=dev)
b_nd = tvm.runtime.tensor(b_np, device=dev)
c_nd = tvm.runtime.empty(c_mm_relu.shape, dtype, dev)


def profile_ms(mod):
    """Return the median time per kernel call in milliseconds."""
    lib = tvm.build(mod, target="llvm")
    timer = lib.time_evaluator("mm_relu", dev, number=100, repeat=5)
    return median(timer(a_nd, b_nd, c_nd).results) * 1_000


In [ ]:
j_factors = [1,2,4,8,16,32,64,128]

baseline_ms = profile_ms(MyModule)
print(f"{'j_factor':>10} {'median (ms)':>14} {'speedup':>10}")
print(f"{'baseline':>10} {baseline_ms:14.4f} {'1.00x':>10}")

for j_factor in j_factors:
    scheduled = transform(MyModule, j_factor)
    elapsed_ms = profile_ms(scheduled)
    print(f"{j_factor:10d} {elapsed_ms:14.4f} {baseline_ms / elapsed_ms:9.2f}x")


  j_factor    median (ms)    speedup
  baseline         3.7445      1.00x
         1         2.9627      1.26x
         2         1.7178      2.18x
         4         0.7911      4.73x
         8         0.4841      7.73x
        16         0.3321     11.28x
        32         0.2881     13.00x
        64         0.3100     12.08x
       128         0.3085     12.14x


## 2.5 Exercises

In [ ]:
import IPython
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import tirx as T

# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(16, 0, -1).reshape(4, 4)
c_np = a + b
print("a=", a, "\nb=", b, "\nc_np=", c_np)

a= [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]] 
b= [[16 15 14 13]
 [12 11 10  9]
 [ 8  7  6  5]
 [ 4  3  2  1]] 
c_np= [[16 16 16 16]
 [16 16 16 16]
 [16 16 16 16]
 [16 16 16 16]]


### 2.5.1.2 Broadcast Add

In [ ]:
# TODO

The scaffold below is for Broadcast Add. Fill in the TensorIR body, then uncomment the NumPy check.

In [ ]:
def build_and_check(mod, symbol, arrays, expected, device):
    lib = tvm.build(mod, target="llvm")
    inputs = tuple(tvm.runtime.tensor(array, device=device) for array in arrays)
    out = tvm.runtime.empty(expected.shape, str(expected.dtype), device)
    args = (*inputs, out)
    lib[symbol](*args)
    if np.issubdtype(expected.dtype, np.integer):
        np.testing.assert_array_equal(out.numpy(), expected)
    else:
        np.testing.assert_allclose(out.numpy(), expected, rtol=1e-4, atol=1e-4)
    return lib, args


exercise_cpu = tvm.cpu(0)
add_a = np.arange(16, dtype="int64").reshape(4, 4)
add_b = np.arange(4, 0, -1, dtype="int64")
add_expected = add_a + add_b


@tvm.script.ir_module
class MyAdd:
    @T.prim_func(s_tir=True)
    def add(A: T.Buffer((4, 4), "int64"),
            B: T.Buffer((4,), "int64"),
            C: T.Buffer((4, 4), "int64")):
        T.func_attr({"global_symbol": "add", "tirx.noalias": True})
        # TODO: Loop over the output and define the spatial axes.
        # TODO: Index the one-dimensional operand with the broadcast rule.
        T.evaluate(0)  # Replace this placeholder with your block.


# Uncomment after completing the TODOs.
# _ = build_and_check(MyAdd, "add", (add_a, add_b), add_expected, exercise_cpu)


### 2.5.1.3 2D Convolution

Use NCHW layout, stride 1 and no padding. Two 3 × 3 filters give an output of shape `(1, 2, 6, 6)`.

In [ ]:
conv_data = np.arange(64, dtype="int64").reshape(1, 1, 8, 8)
conv_weight = np.arange(18, dtype="int64").reshape(2, 1, 3, 3)
conv_windows = np.lib.stride_tricks.sliding_window_view(
    conv_data, (3, 3), axis=(-2, -1),
)
conv_expected = np.einsum("nchwij,ocij->nohw", conv_windows, conv_weight)


@tvm.script.ir_module
class MyConv:
    @T.prim_func(s_tir=True)
    def conv(A: T.Buffer((1, 1, 8, 8), "int64"),
             W: T.Buffer((2, 1, 3, 3), "int64"),
             C: T.Buffer((1, 2, 6, 6), "int64")):
        T.func_attr({"global_symbol": "conv", "tirx.noalias": True})
        # TODO: Set up output loops and reduction axes for the filter.
        # TODO: Initialize each output and accumulate the indexed products.
        T.evaluate(0)  # Replace this placeholder with your reduction block.


# Uncomment after completing the TODOs.
# _ = build_and_check(
#     MyConv, "conv", (conv_data, conv_weight), conv_expected, exercise_cpu,
# )


### 2.5.2.2 Batch Matmul + ReLU

Write the baseline for 16 independent 128 × 128 matrix products, then use the [book’s target schedule](https://book.mlc.ai/chapter_tensor_program/tensorir_exercises.html#exercise-3-transform-a-batch-matmul-program) to guide your transformations.

In [ ]:
bmm_rng = np.random.default_rng(0)
bmm_a = bmm_rng.standard_normal((16, 128, 128), dtype=np.float32)
bmm_b = bmm_rng.standard_normal((16, 128, 128), dtype=np.float32)
bmm_expected = np.maximum(bmm_a @ bmm_b, 0)


@tvm.script.ir_module
class MyBmmRelu:
    @T.prim_func(s_tir=True)
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"),
                 B: T.Buffer((16, 128, 128), "float32"),
                 C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"global_symbol": "bmm_relu", "tirx.noalias": True})
        # TODO: Allocate Y and write the batched reduction in a block named "Y".
        # TODO: In a second block named "C", read Y and write the ReLU result.
        T.evaluate(0)  # Replace this placeholder with your blocks.


# Uncomment to check the baseline after writing the two blocks.
# _ = build_and_check(
#     MyBmmRelu, "bmm_relu", (bmm_a, bmm_b), bmm_expected, exercise_cpu,
# )


In [ ]:
def schedule_bmm_relu(mod):
    sch = tvm.s_tir.Schedule(mod)
    # TODO: Get blocks "Y" and "C" and the loops to transform.
    # TODO: Match the target's j tiles of 8 and k groups of 4.
    # TODO: Place "C" per tile and separate the reduction initialization.
    # TODO: Match the target's parallel, vectorized, and unrolled loops.
    return sch.mod


# Uncomment after filling in the schedule.
# scheduled_bmm = schedule_bmm_relu(MyBmmRelu)
# print(scheduled_bmm.script())


### 2.5.2.3 Build and Evaluate

Check both kernels against NumPy, then compare median runtime with the same inputs and timer settings.

In [ ]:
def compare_bmm_relu(before, after):
    times_ms = []
    for label, mod in (("Original", before), ("Scheduled", after)):
        lib, args = build_and_check(
            mod, "bmm_relu", (bmm_a, bmm_b), bmm_expected, exercise_cpu,
        )
        timer = lib.time_evaluator("bmm_relu", exercise_cpu, number=10, repeat=5)
        elapsed_ms = float(np.median(timer(*args).results)) * 1_000
        times_ms.append(elapsed_ms)
        print(f"{label:>9}: {elapsed_ms:.3f} ms")
    print(f"Speedup: {times_ms[0] / times_ms[1]:.2f}x")


# TODO: Uncomment after validating the baseline and building scheduled_bmm.
# compare_bmm_relu(MyBmmRelu, scheduled_bmm)
